In [1]:
import os
from pathlib import Path
from PIL import Image
import json

TARGET_SIZE = (512, 512)
CAPTION = "a photo of a <sksdog> dog"

def prepare_dataset(golden_folder: str, output_folder: str):
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(os.path.join(output_folder, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_folder, "metadata"), exist_ok=True)

    image_extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    images = sorted([f for f in os.listdir(golden_folder) if any(f.endswith(ext) for ext in image_extensions)])

    print(f"There are {len(images)} images")

    metadata = {
        "dataset_name": "golden_retrievers",
        "total_images": len(images),
        "resolution": "512x512",
        "images": []
    }

    for idx, image_file in enumerate(images, 1):
        try:
            img_path = os.path.join(golden_folder, image_file)
            img = Image.open(img_path).convert("RGB")

            original_size = img.size
            if img.size != TARGET_SIZE:
                img = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)

            out_img = os.path.join(output_folder, "images", f"golden_{idx:04d}.jpg")
            img.save(out_img, "JPEG", quality=95)

            txt_path = os.path.join(output_folder, "images", f"golden_{idx:04d}.txt")
            with open(txt_path, "w") as f:
                f.write(CAPTION)

            metadata["images"].append({
                "id": f"golden_{idx:04d}",
                "filename": f"golden_{idx:04d}.jpg",
                "original_name": image_file,
                "original_size": original_size,
                "final_size": [512, 512],
                "caption": CAPTION
            })

        except Exception as e:
            print(f"Error with {image_file}: {e}")

    with open(os.path.join(output_folder, "metadata", "dataset_metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved images in: {output_folder}/images")
    print(f"Saved metadata in: {output_folder}/metadata/dataset_metadata.json")


if __name__ == "__main__":
    golden_folder = r"C:\Users\Usuario\OneDrive\Escritorio\LORA\Golden"
    output_folder = r"C:\Users\Usuario\OneDrive\Escritorio\LORA\Golden_Output"
    prepare_dataset(golden_folder, output_folder)


There are 113 images
Saved images in: C:\Users\Usuario\OneDrive\Escritorio\LORA\Golden_Output/images
Saved metadata in: C:\Users\Usuario\OneDrive\Escritorio\LORA\Golden_Output/metadata/dataset_metadata.json


In [2]:
import torch
print(torch.cuda.is_available())  # DEBE ser True
print(torch.cuda.get_device_name(0))


True
NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import StableDiffusionPipeline
from diffusers.optimization import get_cosine_schedule_with_warmup
from peft import get_peft_model, LoraConfig
from accelerate import Accelerator
from tqdm import tqdm
import torchvision.transforms as T
import time

TARGET_SIZE = 384   # 512 puede romper en 4GB, 384 es más seguro
start_time = time.time()

class GoldenDataset(Dataset):
    def __init__(self, images_dir):
        self.images = sorted([f for f in os.listdir(images_dir) if f.endswith(".jpg")])
        self.images_dir = images_dir

        self.transform = T.Compose([
            T.Resize((TARGET_SIZE, TARGET_SIZE)),
            T.ToTensor(),
            T.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img = Image.open(os.path.join(self.images_dir, img_name)).convert("RGB")
        img = self.transform(img)

        txt = img_name.replace(".jpg", ".txt")
        with open(os.path.join(self.images_dir, txt), encoding="utf-8") as f:
            caption = f.read().strip()

        return {"image": img, "caption": caption}


def train_lora(
    images_dir,
    output_dir,
    rank=8,
    batch_size=1,
    grad_acc=4,
    steps=2000,
    lr=1e-4
):
    accelerator = Accelerator(
        gradient_accumulation_steps=grad_acc,
        mixed_precision="no"
    )


    device = accelerator.device

    print("🔹 Cargando pipeline base...")
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float32,
        safety_checker=None
    )

    pipe.enable_model_cpu_offload()
    pipe.unet.enable_gradient_checkpointing()

    # Token especial
    if "<sksdog>" not in pipe.tokenizer.get_vocab():
        pipe.tokenizer.add_tokens("<sksdog>")
        pipe.text_encoder.resize_token_embeddings(len(pipe.tokenizer))

    # Freeze VAE y text encoder
    for p in pipe.vae.parameters(): p.requires_grad = False
    for p in pipe.text_encoder.parameters(): p.requires_grad = False

    # LoRA config
    lora_config = LoraConfig(
        r=rank,
        lora_alpha=rank*2,
        target_modules=["to_q","to_k","to_v","to_out.0"],
        lora_dropout=0.05,
        bias="none"
    )

    unet = get_peft_model(pipe.unet, lora_config)
    unet.train()

    optimizer = torch.optim.AdamW(unet.parameters(), lr=lr)

    dataset = GoldenDataset(images_dir)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    lr_scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=100,
        num_training_steps=steps
    )

    unet, optimizer, dataloader, lr_scheduler = accelerator.prepare(
        unet, optimizer, dataloader, lr_scheduler
    )

    noise_scheduler = pipe.scheduler

    global_step = 0
    pbar = tqdm(total=steps)

    print("🚀 Entrenando...")

    while global_step < steps:
        for batch in dataloader:
            with accelerator.accumulate(unet):

                images = batch["image"].to(device, dtype=torch.float32)
                captions = batch["caption"]

                # --- VAE y texto en CPU ---
                with torch.no_grad():
                    latents = pipe.vae.encode(images).latent_dist.sample() * 0.18215

                    text_inputs = pipe.tokenizer(
                    captions,
                    padding="max_length",
                    max_length=77,
                    return_tensors="pt"
                    ).to(device)

                    text_emb = pipe.text_encoder(text_inputs.input_ids)[0]


                # Pasar a GPU solo lo necesario
                latents = latents.to(device, dtype=torch.float32)
                text_emb = text_emb.to(device, dtype=torch.float32)

                noise = torch.randn_like(latents, dtype=torch.float32)
                timesteps = torch.randint(
                    0,
                    noise_scheduler.config.num_train_timesteps,
                    (latents.size(0),),
                    device=device,
                    dtype=torch.long
                )

                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                noise_pred = unet(noisy_latents, timesteps, text_emb).sample
                loss = torch.nn.functional.mse_loss(noise_pred, noise)

                accelerator.backward(loss)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            global_step += 1
            pbar.update(1)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

            if global_step % 100 == 0:
                elapsed = time.time() - start_time
                print(f"\n⏱️ Paso {global_step}/{steps} — {elapsed/60:.1f} min")

            if global_step >= steps:
                break

    accelerator.unwrap_model(unet).save_pretrained(output_dir)
    print(f"\n✅ LoRA guardado en: {output_dir}")


if __name__ == "__main__":
    base = r"C:\Users\Usuario\OneDrive\Escritorio\LORA\Golden_Output\images"

    for r in [8, 16]:
        train_lora(
            images_dir=base,
            output_dir=f"models/lora_sksdog_r{r}",
            rank=r
        )

    total = time.time() - start_time
    print(f"\n🏁 Tiempo total: {total/60:.1f} minutos")


🔹 Cargando pipeline base...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .

  0%|                                                                                         | 0/2000 [00:00<?, ?it/s]

🚀 Entrenando...



  5%|███▏                                                            | 100/2000 [15:41<4:05:19,  7.75s/it, loss=0.0173]

...


⏱️ Paso 1900/2000 — 550.7 min



100%|███████████████████████████████████████████████████████████████| 2000/2000 [5:07:59<00:00,  8.73s/it, loss=0.1843]


⏱️ Paso 2000/2000 — 566.6 min

✅ LoRA guardado en: models/lora_sksdog_r16


100%|███████████████████████████████████████████████████████████████| 2000/2000 [5:08:00<00:00,  9.24s/it, loss=0.1843]


🏁 Tiempo total: 566.6 minutos


In [1]:
import torch
from diffusers import StableDiffusionPipeline
import os
from peft import PeftModel

BASE_MODEL = "runwayml/stable-diffusion-v1-5"
LORA_DIR = r"C:\Users\Usuario\OneDrive\Escritorio\LORA\models"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16
).to(device)

#pipe.enable_xformers_memory_efficient_attention()  xformers es equis de que flipas

prompts = [
    "a photo of a <sksdog> dog in a park",
    "a <sksdog> dog sitting",
    "a <sksdog> dog playing"
]

def generate(pipe, name):
    for i, prompt in enumerate(prompts):
        image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
        image.save(f"{OUTPUT_DIR}/{name}_{i+6}.png")

# Base model
generate(pipe, "base")

# Con LoRAs
for lora in os.listdir(LORA_DIR):
    lora_path = os.path.join(LORA_DIR, lora)
    print(f"Cargando {lora_path}")
    pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)
    generate(pipe, lora)
    pipe.unload_lora_weights()


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

Cargando C:\Users\Usuario\OneDrive\Escritorio\LORA\models\lora_sksdog_r8


C:\Users\Usuario\anaconda3\envs\lora_env_sinx\lib\site-packages\peft\tuners\tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
C:\Users\Usuario\anaconda3\envs\lora_env_sinx\lib\site-packages\peft\peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.lora_A.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.lora_B.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.lora_A.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.lora_B.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [1]:
import torch
from diffusers import StableDiffusionPipeline
import os
import time
from peft import PeftModel

BASE_MODEL = "runwayml/stable-diffusion-v1-5"
LORA_DIR = r"C:\Users\Usuario\OneDrive\Escritorio\LORA\models"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16
).to(device)

# =========================
# Utilidades de medición
# =========================

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

prompts = [
    "a photo of a <sksdog> dog in a park",
    "a <sksdog> dog sitting",
    "a <sksdog> dog"
]

# =========================
# Generación instrumentada
# =========================

def generate(pipe, name):
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start_time = time.time()

    with torch.no_grad():
        for i, prompt in enumerate(prompts):
            image = pipe(
                prompt,
                num_inference_steps=30,
                guidance_scale=7.5
            ).images[0]
            image.save(f"{OUTPUT_DIR}/{name}_{i}.png")

    torch.cuda.synchronize()
    elapsed = time.time() - start_time
    peak_vram = torch.cuda.max_memory_allocated() / 1024**2

    return elapsed, peak_vram

# =========================
# MODELO BASE
# =========================

print("\n=== BASE MODEL ===")

base_trainable = count_trainable_params(pipe.unet)
base_total = count_total_params(pipe.unet)

print(f"Trainable params: {base_trainable:,}")
print(f"Total params: {base_total:,}")
print(f"Trainable ratio: {100 * base_trainable / base_total:.6f}%")

base_time, base_vram = generate(pipe, "base")

# =========================
# MODELOS LoRA
# =========================

results = []

for lora in os.listdir(LORA_DIR):
    lora_path = os.path.join(LORA_DIR, lora)
    print(f"\n=== Cargando LoRA: {lora} ===")

    pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)

    lora_trainable = count_trainable_params(pipe.unet)
    lora_total = count_total_params(pipe.unet)

    print(f"Trainable params: {lora_trainable:,}")
    print(f"Total params: {lora_total:,}")
    print(f"Trainable ratio: {100 * lora_trainable / lora_total:.6f}%")

    time_inf, vram_inf = generate(pipe, lora)

    results.append({
        "model": lora,
        "trainable_params": lora_trainable,
        "trainable_ratio_pct": 100 * lora_trainable / lora_total,
        "inference_time_sec": time_inf,
        "peak_vram_gb": vram_inf
    })

    pipe.unload_lora_weights()

# =========================
# RESUMEN FINAL
# =========================

import pandas as pd

df = pd.DataFrame(results)
print("\n=== SUMMARY ===")
print(df)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!



=== BASE MODEL ===
Trainable params: 859,520,964
Total params: 859,520,964
Trainable ratio: 100.000000%


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

C:\Users\Usuario\anaconda3\envs\lora_env_sinx\lib\site-packages\peft\peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.lora_A.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.lora_B.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.lora_A.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.lora_B.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v.lora_A.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v.lora_B.default.weight', 'base_model.model.base_model.model.down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.lora_A.default.weight', 'base_model

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]


=== SUMMARY ===
             model  trainable_params  trainable_ratio_pct  inference_time_sec  \
0  lora_sksdog_r16                 0                  0.0           25.433293   
1   lora_sksdog_r8                 0                  0.0           28.854697   

   peak_vram_gb  
0   3268.729492  
1   3262.647461  
